# 第1章 5分でLLMと話す

Google Colabで、本物のLLMを0円で動かす

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/01-first-llm.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。

**このノートはGPUを使います。** メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選んでから実行してください。


## 最初の1行を実行する


In [ ]:
print("こんにちは、LLM！")

In [ ]:
greeting = "ボン"
print(greeting * 3 + "ゴ！")

## 無料のGPUを借りる


In [ ]:
!nvidia-smi

## LLMを呼び出す


In [ ]:
import torch
from transformers import pipeline

gpu = torch.cuda.is_available()   # GPUが使えるなら True、使えないなら False

chat = pipeline(
    "text-generation",                         # やりたいこと：文章を生成する
    model="Qwen/Qwen2.5-0.5B-Instruct",        # 使うモデルの名前
    dtype=torch.float16 if gpu else torch.float32,  # 数の細かさ（GPUなら軽い形式）
    device_map="auto",                         # GPUがあればGPUに載せる
)
print("GPUを使う？", gpu)

## 話しかけてみる


In [ ]:
messages = [
    {"role": "user", "content": "こんにちは！あなたは誰ですか？"},
]
result = chat(messages, max_new_tokens=100)
print(result[0]["generated_text"][-1]["content"])

## 毎回書くのは面倒なので、関数にする


In [ ]:
def ask(question, max_new_tokens=200):
    messages = [{"role": "user", "content": question}]
    result = chat(messages, max_new_tokens=max_new_tokens)
    return result[0]["generated_text"][-1]["content"]

print(ask("朝ごはんにおすすめの食べ物を3つ、理由つきで教えて"))

In [ ]:
print(123 * 456)

## LLMの正体：次の言葉を当てるゲームの達人


In [ ]:
out = chat("吾輩は猫である。名前は", max_new_tokens=30, do_sample=False)
print(out[0]["generated_text"])

In [ ]:
messages = [
    {"role": "system", "content": "あなたは陽気な物理の先生です。"},
    {"role": "user", "content": "空はなぜ青いの？"},
]
script = chat.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(script)

## キャラクターを演じさせる


In [ ]:
def ask_as(character, question, max_new_tokens=200):
    messages = [
        {"role": "system", "content": character},
        {"role": "user", "content": question},
    ]
    result = chat(messages, max_new_tokens=max_new_tokens)
    return result[0]["generated_text"][-1]["content"]

bongo = "あなたはボンゴ先生です。陽気な物理の先生で、答えに必ずダジャレを1つ入れます。"
print(ask_as(bongo, "空はなぜ青いの？"))

## 「大規模」ってどれくらい？


In [ ]:
n = sum(p.numel() for p in chat.model.parameters())
print(f"パラメータの数：{n:,} 個")